In [ ]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms
import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

import time

lib_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Code'
os.chdir(lib_dir)
print('library directory: ' + lib_dir)
from modules.networks_clf import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *
from modules.metrics import accu_fun

library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code


In [ ]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-fbbae4f8-920c-e8be-3c9b-b2ab13fe74b0)


In [ ]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [ ]:
# @title Load data
file_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/'
brain_signal_lfp = torch.load(file_dir + '/brain_signal_lfp1.pt')
for file_id in [2, 3, 4, 5]:
    brain_signal_lfp = torch.concat([brain_signal_lfp, torch.load(file_dir + f'/brain_signal_lfp{file_id}.pt')], dim=0)
    print(f'Load file id: {file_id}')

# torch.save(brain_signal_lfp, '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/brain_signal_lfp')
list_dict = torch.load(file_dir + '/list_dict.pt')
# list_dict_acronym_selec = torch.load(file_dir + '/list_dict_acronym_selec.pt')
brain_region_index = list_dict['brain_region_index']
brain_region_index_Cosmos = list_dict['brain_region_index_Cosmos']
coordinate_list = list_dict['coordinate_list']


Load file id: 2
Load file id: 3
Load file id: 4
Load file id: 5


In [ ]:
if len(brain_signal_lfp) == len(brain_region_index):
    print('Matched, no damage!')

Matched, no damage!


In [ ]:
brain_region_index.type()

'torch.LongTensor'

In [ ]:
id_community_mapping = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Project44/Result/id_community_mapping.pt')
brain_region_index_AnyNet = torch.tensor(id_community_mapping['AnyNet'][brain_region_index], dtype=torch.long)
brain_region_index_ViT = torch.tensor(id_community_mapping['ViT'][brain_region_index], dtype=torch.long)
brain_region_index_RNN = torch.tensor(id_community_mapping['RNN'][brain_region_index], dtype=torch.long)

In [ ]:
save_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44'

In [ ]:
list_dict.keys()

dict_keys(['brain_region_index', 'brain_region_index_Cosmos', 'brain_region_atlas', 'subject_list', 'exp_list', 'key_list', 'coordinate_list', 'depth_list', 'volume_list', 'brain_signal_lfp', 'brain_signal_ap', 'train_list_key', 'test_list_key', 'train_list_subject', 'test_list_subject', 'train_list_exp', 'test_list_exp', 'train_list_trial', 'test_list_trial', 'train_list_intest', 'test_list_intest', 'acronym_selec_list', 'valid_list_intest'])

In [ ]:
key = 'stimOff_times'
model_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Model/Allen'
subject_od_ind = torch.load(model_dir + f'/subject_od_ind_Allen_{key}{0}.pt')
subject_od_list = torch.load(model_dir + f'/subject_od_list_Allen_{key}{0}.pt')
train_ind, valid_ind, test_ind, test_subject_ind = dat_ind_gen(list_dict, subject_od_ind, key)

In [ ]:
def iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index0, coordinate_list):
    data_train = TensorDataset(brain_signal_lfp[train_ind,:], brain_region_index0[train_ind], coordinate_list[train_ind])
    data_valid = TensorDataset(brain_signal_lfp[valid_ind,:], brain_region_index0[valid_ind], coordinate_list[valid_ind])
    data_test = TensorDataset(brain_signal_lfp[test_ind,:], brain_region_index0[test_ind], coordinate_list[test_ind])

    train_iter = DataLoader(data_train, batch_size=128, shuffle=True)
    valid_iter = DataLoader(data_valid, batch_size=128, shuffle=True)
    test_iter = DataLoader(data_test, batch_size=128, shuffle=True)

    return train_iter, valid_iter, test_iter

In [ ]:
# @title Training function
def net_train_AnyNet_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    time_array_train = []
    time_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        epoch0 = 0
        time0_train = time.time()
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                if epoch0 == 0 and epoch == 0:
                    if accu_fun(py_train, y_train) == (torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu():
                        print('accu_fun is correct>>>>>>>>>>>>>>>>>>>>>>>>>>>>')
                    else:
                        print('accu_fun is wrong>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>')
                        return
                print(accu_fun(py_train, y_train))
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append(accu_fun(py_train, y_train))
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))
        time_array_train.append(time.time() - time0_train)

        Classifier.eval()
        acu_valid = []
        time0_valid = time.time()
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append(accu_fun(py_valid, y_valid))
            # acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/Community/AnyNet_L_AllenCommunity_{key}_{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/Community/AnyNet_L_AllenCommunity_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))

        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))
        time_array_valid.append(time.time() - time0_valid)

    torch.save(acu_array_train, train_args['save_dir'] + f'/Model/Community/AnyNet_L_AllenCommunity_{key}_train_acu{ind}.pt')
    torch.save(acu_array_valid, train_args['save_dir'] + f'/Model/Community/AnyNet_L_AllenCommunity_{key}_valid_acu{ind}.pt')

    return


def net_train_ViT_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    time_array_train = []
    time_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        epoch0 = 0
        time0_train = time.time()
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))
        time_array_train.append(time.time() - time0_train)

        Classifier.eval()
        acu_valid = []
        time0_valid = time.time()
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/Community/ViT_L_AllenCommunity_{key}_{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/Community/ViT_L_AllenCommunity_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))

        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))
        time_array_valid.append(time.time() - time0_valid)

    torch.save(torch.tensor(acu_array_train), train_args['save_dir'] + f'/Model/Community/ViT_L_AllenCommunity_{key}_train_acu{ind}.pt')
    torch.save(torch.tensor(acu_array_valid), train_args['save_dir'] + f'/Model/Community/ViT_L_AllenCommunity_{key}_valid_acu{ind}.pt')

    return

def net_train_RNN_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    time_array_train = []
    time_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        time0_train = time.time()
        epoch0 = 0
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args).squeeze(1).permute(0, 2, 1)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))
        time_array_train.append(time.time() - time0_train)

        Classifier.eval()
        acu_valid = []
        time0_valid = time.time()
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args).squeeze(1).permute(0, 2, 1)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/Community/RNN_L_AllenCommunity_{key}_{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/Community/RNN_L_AllenCommunity_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))


        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))
        time_array_valid.append(time.time() - time0_valid)

    torch.save(torch.tensor(acu_array_train), train_args['save_dir'] + f'/Model/Community/RNN_L_AllenCommunity_{key}_train_acu{ind}.pt')
    torch.save(torch.tensor(acu_array_valid), train_args['save_dir'] + f'/Model/Community/RNN_L_AllenCommunity_{key}_valid_acu{ind}.pt')

    return


In [ ]:
train_iter_AnyNet, valid_iter_AnyNet, test_iter_AnyNet = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_AnyNet, coordinate_list)
# train_iter_ViT, valid_iter_ViT, test_iter_ViT = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_ViT, coordinate_list)
# train_iter_RNN, valid_iter_RNN, test_iter_RNN = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_RNN, coordinate_list)

In [ ]:
# @title AnyNet model definition
c0 = 64 * 4
k0 = 1.0

model_args = {
    'arch':((2,c0 * 2,1,k0), (2,c0 * 3,1,k0), (2,c0 * 4,1,k0), (2,c0 * 5,1,k0)),
    'stem_channels':c0,
}
train_args = {
    'overfitting_thres':0.60,
    'lr':5e-4,
    'norm':True,
    'temp':[True, True],
    'epochs':10,
    'save_dir':save_dir,
}



In [ ]:
for ind in range(0, 1):
    Classifier = AnyNet_L(model_args['arch'], model_args['stem_channels'], frequency_bin=spectro_args['spectro_img'][0], num_classes=len(acronym_list)).to(device)
    Classifier.apply(init_cnn)
    net_train_AnyNet_L(train_iter_AnyNet, valid_iter_AnyNet, Classifier.to(device), spectro_args, train_args, key, ind, device)

accu_fun is correct>>>>>>>>>>>>>>>>>>>>>>>>>>>>
tensor(0.)
tensor(0.3359)
tensor(0.4297)
tensor(0.4141)
tensor(0.5312)
tensor(0.4453)
tensor(0.5078)
tensor(0.5312)
tensor(0.5703)
tensor(0.6094)
tensor(0.6094)
tensor(0.5312)
tensor(0.5859)
Acu Train: 0.4987219274044037
Acu valid: 0.5371237397193909
tensor(0.6797)
tensor(0.6016)
tensor(0.5625)
tensor(0.6172)
tensor(0.6875)
tensor(0.6641)
tensor(0.6250)
tensor(0.6953)
tensor(0.6328)
tensor(0.6562)
tensor(0.6328)
tensor(0.6641)
tensor(0.7344)
Acu Train: 0.6388188004493713
Acu valid: 0.5784070491790771
tensor(0.7344)
tensor(0.6797)
tensor(0.6406)
tensor(0.7734)
tensor(0.7344)
tensor(0.6719)
tensor(0.7031)
tensor(0.7109)
tensor(0.7266)
tensor(0.7188)
tensor(0.6797)
tensor(0.7109)
tensor(0.7344)
Acu Train: 0.7139418125152588
Acu valid: 0.5887327194213867
tensor(0.7500)
tensor(0.7656)
tensor(0.6484)
tensor(0.7891)
tensor(0.7812)
tensor(0.7891)
tensor(0.7734)
tensor(0.7188)
tensor(0.8281)
tensor(0.7891)
tensor(0.8594)
tensor(0.7969)
tensor(0.76

In [ ]:
# del train_iter_ViT, valid_iter_ViT, test_iter_ViT

In [ ]:
# del train_iter_AnyNet, valid_iter_AnyNet, test_iter_AnyNet
train_iter_ViT, valid_iter_ViT, test_iter_ViT = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_ViT, coordinate_list)

In [39]:
train_args['epochs'] = 50
for ind in range(0, 1):
    img_size, patch_size = (224, 28), (28, 28)
    num_hiddens, mlp_num_hiddens, num_heads, num_blks = 512, 2048, 8, 4
    emb_dropout, blk_dropout = 0.1, 0.1
    Classifier = ViT_L(spectro_args['spectro_img'][0], img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads, num_blks, emb_dropout, blk_dropout, num_classes=len(acronym_list)).to(device)
    net_train_ViT_L(train_iter_ViT, valid_iter_ViT, Classifier.to(device), spectro_args, train_args, key, ind, device)

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


tensor(0.)
tensor(0.4766)
tensor(0.3984)
tensor(0.3203)
tensor(0.5000)
tensor(0.3906)
tensor(0.4688)
tensor(0.4141)
tensor(0.4219)
tensor(0.3906)
tensor(0.4375)
tensor(0.4609)
tensor(0.3594)
Acu Train: 0.4126878082752228
Acu valid: 0.45945316553115845
tensor(0.4141)
tensor(0.3828)
tensor(0.4609)
tensor(0.4844)
tensor(0.4844)
tensor(0.4688)
tensor(0.4922)
tensor(0.3828)
tensor(0.5000)
tensor(0.4297)
tensor(0.5000)
tensor(0.4609)
tensor(0.4609)
Acu Train: 0.4837232828140259
Acu valid: 0.518089234828949
tensor(0.5156)
tensor(0.5547)
tensor(0.4375)
tensor(0.4297)
tensor(0.5000)
tensor(0.5547)
tensor(0.4922)
tensor(0.5781)
tensor(0.5625)
tensor(0.5312)
tensor(0.6250)
tensor(0.5703)
tensor(0.4688)
Acu Train: 0.5057061314582825
Acu valid: 0.4924095571041107
tensor(0.4922)
tensor(0.4297)
tensor(0.3672)
tensor(0.4453)
tensor(0.5000)
tensor(0.5312)
tensor(0.5547)
tensor(0.5078)
tensor(0.5859)
tensor(0.5781)
tensor(0.5078)
tensor(0.5781)
tensor(0.5078)
Acu Train: 0.5051523447036743
Acu valid: 0.5

In [40]:
del train_iter_ViT, valid_iter_ViT, test_iter_ViT
train_iter_RNN, valid_iter_RNN, test_iter_RNN = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_RNN, coordinate_list)
train_args['epochs'] = 30

In [41]:
RNN_args = {
    'input_size':224,
    'hidden_size':512 * 2,
    'num_layers':2,
    'category_num':len(acronym_list),
    'data_len':28,
}
for ind in range(0, 1):
    Classifier = RNN_L(spectro_args['spectro_img'][0], RNN_args).to(device)
    net_train_RNN_L(train_iter_RNN, valid_iter_RNN, Classifier.to(device), spectro_args, train_args, key, ind, device)

tensor(0.)
tensor(0.3125)
tensor(0.3906)
tensor(0.3672)
tensor(0.3750)
tensor(0.3984)
tensor(0.5312)
tensor(0.4609)
tensor(0.5391)
tensor(0.5391)
tensor(0.5156)
tensor(0.5078)
tensor(0.4766)
Acu Train: 0.4393385052680969
Acu valid: 0.5307941436767578
tensor(0.4844)
tensor(0.5547)
tensor(0.6016)
tensor(0.5156)
tensor(0.5234)
tensor(0.5391)
tensor(0.6484)
tensor(0.5938)
tensor(0.6250)
tensor(0.6172)
tensor(0.6484)
tensor(0.5781)
tensor(0.7109)
Acu Train: 0.5877593755722046
Acu valid: 0.6023703813552856
tensor(0.7266)
tensor(0.6719)
tensor(0.6484)
tensor(0.6562)
tensor(0.6484)
tensor(0.7266)
tensor(0.7031)
tensor(0.6406)
tensor(0.7188)
tensor(0.6875)
tensor(0.6250)
tensor(0.6484)
tensor(0.7500)
Acu Train: 0.6625499129295349
Acu valid: 0.6173833608627319
tensor(0.6641)
tensor(0.8125)
tensor(0.6172)
tensor(0.6719)
tensor(0.7578)
tensor(0.7656)
tensor(0.7109)
tensor(0.6641)
tensor(0.7109)
tensor(0.7422)
tensor(0.6875)
tensor(0.6953)
tensor(0.6562)
Acu Train: 0.7045792937278748
Acu valid: 0.6

In [ ]:
from google.colab import runtime
runtime.unassign()